# 🏦 Banking Intent Detection - Fine-tuning with Unsloth

**Dataset:** BANKING77 (PolyAI)  
**Model:** LLaMA-3-8B (4-bit quantized via Unsloth)  
**Platform:** Kaggle (Tesla T4 GPU)

## 1. Xóa cache cũ & Cài đặt thư viện

> ⚠️ **Phải chạy cell này đầu tiên và RESTART SESSION sau đó!**

In [1]:
import os, shutil

# BƯỚC 1: Xóa Unsloth compiled cache cũ (nguyên nhân gây lỗi 'int has no attribute mean')
cache_path = "/kaggle/working/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"✅ Đã xóa cache cũ: {cache_path}")
else:
    print("Cache không tồn tại, bỏ qua.")

print("\nCache đã được dọn dẹp. Tiếp tục cài đặt thư viện...")

Cache không tồn tại, bỏ qua.

Cache đã được dọn dẹp. Tiếp tục cài đặt thư viện...


In [2]:
%%capture
# BƯỚC 2: Gỡ và cài lại thư viện sạch
!pip uninstall -y unsloth transformers trl peft accelerate bitsandbytes unsloth_zoo

# Transformers >=4.47 <5.0
!pip install --upgrade --no-cache-dir "transformers>=4.47.0,<5.0.0"

# Cài Unsloth mới nhất
!pip install --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir unsloth_zoo trl peft accelerate bitsandbytes datasets scikit-learn pyarrow fastparquet

print("\n✅ Cài đặt xong!")

In [3]:
# BƯỚC 3: Restart kernel để nạp thư viện mới
print("🔄 Đang restart kernel...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

🔄 Đang restart kernel...


{'status': 'ok', 'restart': True}

---
## ✅ Chạy từ đây sau khi kernel đã restart

## 2. Import & Kiểm tra GPU

In [1]:
from unsloth import FastLanguageModel
import torch
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset
from sklearn.metrics import accuracy_score, classification_report
from trl import SFTTrainer
from transformers import TrainingArguments
import os, warnings
warnings.filterwarnings('ignore')

import transformers
print(f"Transformers version: {transformers.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-26 06:45:23.007674: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777185923.031729    1332 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777185923.039989    1332 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777185923.061278    1332 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777185923.061305    1332 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777185923.061308    1332 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Transformers version: 4.57.6
GPU: Tesla T4
VRAM: 14.6 GB


## 3. Cấu hình Hyperparameters

In [5]:
# ===== Model Config =====
MODEL_NAME     = "unsloth/llama-3-8b-bnb-4bit"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True

# ===== LoRA Config =====
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# ===== Training Config =====
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 2
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size = 2 * 8 = 16
EPOCHS         = 1
OPTIMIZER      = "adamw_8bit"
WEIGHT_DECAY   = 0.01
LR_SCHEDULER   = "linear"
SEED           = 3407
WARMUP_STEPS   = 5

# ===== Data Config =====
SAMPLE_RATIO   = 0.5  # Lấy 50% dữ liệu train

# ===== Output =====
OUTPUT_DIR     = "outputs"

print("✅ Cấu hình đã sẵn sàng!")
print(f"   Model: {MODEL_NAME}")
print(f"   LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   LR={LEARNING_RATE}, Batch={BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS}={BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS}")
print(f"   Epochs={EPOCHS}, Optimizer={OPTIMIZER}")
print(f"   Sample ratio={SAMPLE_RATIO}")

✅ Cấu hình đã sẵn sàng!
   Model: unsloth/llama-3-8b-bnb-4bit
   LoRA r=16, alpha=32
   LR=0.0002, Batch=2x8=16
   Epochs=1, Optimizer=adamw_8bit
   Sample ratio=0.5


## 4. Tải dữ liệu BANKING77

In [6]:
# --- Tải dữ liệu ---
train_url = "https://huggingface.co/datasets/PolyAI/banking77/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
test_url  = "https://huggingface.co/datasets/PolyAI/banking77/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet"

try:
    df_train = pd.read_parquet(train_url)
    df_test  = pd.read_parquet(test_url)
    print("✅ Tải dữ liệu thành công!")
except Exception as e:
    print(f"Fallback: {e}")
    ds = load_dataset("PolyAI/banking77")
    df_train = ds['train'].to_pandas()
    df_test  = ds['test'].to_pandas()

# --- Danh sách 77 intent chuẩn (fix lỗi trùng lặp) ---
label_names = [
    'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support',
    'automatic_top_up', 'balance_not_updated_after_bank_transfer',
    'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed',
    'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival',
    'card_delivery_estimate', 'card_linking', 'card_not_working',
    'card_payment_fee_charged', 'card_payment_not_recognised',
    'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge',
    'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card',
    'contactless_not_working', 'country_support', 'declined_card_payment',
    'declined_cash_withdrawal', 'declined_transfer',
    'direct_debit_payment_not_recognised', 'disposable_card_limits',
    'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app',
    'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support',
    'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card',
    'getting_virtual_card', 'lost_or_stolen_card', 'lost_or_stolen_phone',
    'order_physical_card', 'passcode_forgotten', 'pending_card_payment',
    'pending_cash_withdrawal', 'pending_top_up', 'pending_transfer', 'pin_blocked',
    'receiving_money', 'refund_not_showing_up', 'request_refund',
    'reverted_card_payment?', 'supported_cards_and_currencies', 'terminate_account',
    'top_up_by_bank_transfer_charge', 'top_up_by_card_charge',
    'top_up_by_cash_or_cheque', 'top_up_failed', 'top_up_limits', 'top_up_reverted',
    'topping_up_by_card', 'transaction_charged_twice', 'transfer_fee_charged',
    'transfer_into_account', 'transfer_not_received_by_recipient',
    'transfer_timing',  # <-- Fix: trước đây bị trùng với index 66
    'unable_to_verify_identity', 'verify_my_identity', 'verify_source_of_funds',
    'verify_top_up', 'virtual_card_not_working', 'visa_or_mastercard',
    'why_verify_identity', 'wrong_amount_of_cash_received',
    'wrong_exchange_rate_for_cash_withdrawal'
]

assert len(label_names) == 77, f"Expected 77 labels, got {len(label_names)}"

# --- Map label ID → label name ---
if 'label' in df_train.columns and df_train['label'].dtype != object:
    df_train['intent'] = df_train['label'].apply(lambda x: label_names[x] if x < len(label_names) else "unknown")
    df_test['intent']  = df_test['label'].apply(lambda x: label_names[x] if x < len(label_names) else "unknown")
else:
    df_train['intent'] = df_train.get('intent', df_train.get('label'))
    df_test['intent']  = df_test.get('intent', df_test.get('label'))

# --- Sample 50% dữ liệu train (stratified) ---
print(f"\nFull train set: {len(df_train)} samples")
df_train_sampled = df_train.groupby('intent', group_keys=False).apply(
    lambda x: x.sample(frac=SAMPLE_RATIO, random_state=SEED)
).reset_index(drop=True)
print(f"Sampled train set ({int(SAMPLE_RATIO*100)}%): {len(df_train_sampled)} samples")
print(f"Test set: {len(df_test)} samples")
print(f"Number of intents: {len(label_names)}")

# --- Lưu CSV ---
os.makedirs("sample_data", exist_ok=True)
df_train_sampled[['text','intent']].to_csv("sample_data/train.csv", index=False)
df_test[['text','intent']].to_csv("sample_data/test.csv", index=False)
print("\n✅ Dữ liệu đã lưu vào sample_data/")

✅ Tải dữ liệu thành công!

Full train set: 10003 samples
Sampled train set (50%): 4997 samples
Test set: 3080 samples
Number of intents: 77

✅ Dữ liệu đã lưu vào sample_data/


## 5. Tải mô hình & LoRA

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print("✅ Mô hình và LoRA đã sẵn sàng!")

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Mô hình và LoRA đã sẵn sàng!


## 6. Chuẩn bị Dataset cho Training

In [8]:
prompt_template = """Below is an inquiry from a banking customer. Classify the intent of the inquiry.

### Inquiry:
{}

### Intent:
{}"""

def formatting_prompts_func(examples):
    texts = []
    for text, intent in zip(examples["text"], examples["intent"]):
        texts.append(prompt_template.format(text, intent) + tokenizer.eos_token)
    return {"text": texts}

# Sử dụng dữ liệu đã sample
train_dataset = Dataset.from_pandas(df_train_sampled[['text','intent']])
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

print("✅ Dataset hoàn tất!")
print(f"   Số mẫu train: {len(train_dataset)}")
print(f"\n📝 Ví dụ prompt:")
print(train_dataset[0]['text'][:300] + "...")

Map:   0%|          | 0/4997 [00:00<?, ? examples/s]

✅ Dataset hoàn tất!
   Số mẫu train: 4997

📝 Ví dụ prompt:
Below is an inquiry from a banking customer. Classify the intent of the inquiry.

### Inquiry:
What am I going to need in order to activate my card?

### Intent:
activate_my_card<|end_of_text|>...


## 7. Huấn luyện (Training)

> ⚠️ **Lưu ý:** `average_tokens_across_devices=False` là bắt buộc để fix lỗi tương thích giữa Unsloth và Transformers >= 4.57

In [9]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim=OPTIMIZER,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type=LR_SCHEDULER,
        seed=SEED,
        output_dir=OUTPUT_DIR,
        report_to="none",
        average_tokens_across_devices=False,  # FIX: ngăn lỗi 'int has no attribute mean'
    ),
)

print("🚀 Bắt đầu huấn luyện...")
train_result = trainer.train()
print(f"\n✅ Huấn luyện xong!")
print(f"   Training Loss: {train_result.training_loss:.4f}")
print(f"   Total Steps: {train_result.global_step}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4997 [00:00<?, ? examples/s]

🚀 Bắt đầu huấn luyện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,997 | Num Epochs = 1 | Total steps = 157
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,2.324900
20,1.000500
30,0.832300
40,0.792300
50,0.756600
60,0.731600
70,0.752400
80,0.762100
90,0.770600
100,0.712200



✅ Huấn luyện xong!
   Training Loss: 0.8612
   Total Steps: 157


## 8. Lưu Model & Nén

In [11]:
# Lưu LoRA adapter + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model đã lưu vào: {OUTPUT_DIR}/")

# Nén để dễ download
import shutil
shutil.make_archive("outputs", "zip", ".", OUTPUT_DIR)
print("✅ outputs.zip đã sẵn sàng để download!")

# Liệt kê files đã lưu
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"   {f}: {size/1024:.1f} KB")

✅ Model đã lưu vào: outputs/
✅ outputs.zip đã sẵn sàng để download!
   checkpoint-157: 4.0 KB
   README.md: 1.5 KB
   special_tokens_map.json: 0.5 KB
   adapter_model.safetensors: 163898.7 KB
   adapter_config.json: 1.2 KB
   tokenizer_config.json: 49.5 KB
   tokenizer.json: 16806.6 KB


## 9. Đánh giá trên Test Set (Evaluation)

In [12]:
# Chuyển sang chế độ inference
FastLanguageModel.for_inference(model)

inference_prompt = """Below is an inquiry from a banking customer. Classify the intent of the inquiry.

### Inquiry:
{}

### Intent:
"""

def predict(text):
    """Dự đoán intent cho một câu văn bản."""
    inputs = tokenizer([inference_prompt.format(text)], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=32, use_cache=True)
    decoded = tokenizer.batch_decode(outputs)[0]
    try:
        pred = decoded.split("### Intent:")[1]
        # Xử lý các trường hợp end token khác nhau
        for stop_token in [tokenizer.eos_token, "<|end_of_text|>", "<|eot_id|>", "\n\n"]:
            if stop_token in pred:
                pred = pred.split(stop_token)[0]
        return pred.strip().split("\n")[0].strip()
    except (IndexError, AttributeError):
        return "unknown"

# --- Đánh giá trên test set ---
print("📊 Đang đánh giá trên test set...")
print(f"   Số mẫu test: {len(df_test)}")
print("   (Có thể mất 20-40 phút trên T4 GPU)\n")

y_true = []
y_pred = []

for idx, row in df_test.iterrows():
    pred = predict(row['text'])
    y_true.append(row['intent'])
    y_pred.append(pred)
    
    if (idx + 1) % 100 == 0:
        current_acc = accuracy_score(y_true, y_pred)
        print(f"   Processed {idx+1}/{len(df_test)} - Current accuracy: {current_acc:.4f}")

# --- Kết quả ---
accuracy = accuracy_score(y_true, y_pred)
print(f"\n{'='*50}")
print(f"🎯 ACCURACY TRÊN TEST SET: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*50}")

📊 Đang đánh giá trên test set...
   Số mẫu test: 3080
   (Có thể mất 20-40 phút trên T4 GPU)

   Processed 100/3080 - Current accuracy: 0.9300
   Processed 200/3080 - Current accuracy: 0.9300
   Processed 300/3080 - Current accuracy: 0.9033
   Processed 400/3080 - Current accuracy: 0.8950
   Processed 500/3080 - Current accuracy: 0.9100
   Processed 600/3080 - Current accuracy: 0.9100
   Processed 700/3080 - Current accuracy: 0.9000
   Processed 800/3080 - Current accuracy: 0.9087
   Processed 900/3080 - Current accuracy: 0.8944
   Processed 1000/3080 - Current accuracy: 0.8920
   Processed 1100/3080 - Current accuracy: 0.8909
   Processed 1200/3080 - Current accuracy: 0.8917
   Processed 1300/3080 - Current accuracy: 0.8954
   Processed 1400/3080 - Current accuracy: 0.8750
   Processed 1500/3080 - Current accuracy: 0.8760
   Processed 1600/3080 - Current accuracy: 0.8819
   Processed 1700/3080 - Current accuracy: 0.8853
   Processed 1800/3080 - Current accuracy: 0.8817
   Processed 19

In [13]:
# --- Classification Report chi tiết ---
# Chỉ hiển thị các label xuất hiện trong dữ liệu
all_labels = sorted(set(y_true + y_pred))
print("\n📋 Classification Report (top intents):")
print(classification_report(y_true, y_pred, labels=all_labels[:20], zero_division=0))

# Lưu kết quả
results_df = pd.DataFrame({"text": df_test["text"].values, "true_intent": y_true, "predicted_intent": y_pred})
results_df["correct"] = results_df["true_intent"] == results_df["predicted_intent"]
results_df.to_csv("sample_data/test_results.csv", index=False)
print(f"\n✅ Kết quả chi tiết đã lưu vào: sample_data/test_results.csv")


📋 Classification Report (top intents):
                                                  precision    recall  f1-score   support

                                activate_my_card       1.00      0.95      0.97        40
                                       age_limit       1.00      1.00      1.00        40
                         apple_pay_or_google_pay       0.95      1.00      0.98        40
                                     atm_support       0.91      1.00      0.95        40
                                automatic_top_up       0.97      0.90      0.94        40
         balance_not_updated_after_bank_transfer       0.74      0.78      0.76        40
          balance_not_updated_after_card_payment       0.00      0.00      0.00         0
balance_not_updated_after_cheque_or_cash_deposit       0.95      0.90      0.92        40
                         beneficiary_not_allowed       0.84      0.68      0.75        40
                                 cancel_transfer       1.00

## 10. Demo Inference

In [14]:
# --- Demo inference với các câu hỏi mẫu ---
test_messages = [
    "I lost my credit card, how can I freeze it?",
    "Why was I charged a fee for my card payment?",
    "How do I change my PIN?",
    "My transfer hasn't arrived yet.",
    "I want to close my account.",
    "Can I use Apple Pay with my card?",
    "I need a refund for a transaction.",
    "What is the exchange rate for USD?",
]

print("\n" + "="*60)
print("🏦 BANKING INTENT CLASSIFICATION - DEMO")
print("="*60)

for msg in test_messages:
    label = predict(msg)
    print(f"\n💬 {msg}")
    print(f"🏷️  Predicted: {label}")

print("\n" + "="*60)
print("✅ Demo hoàn tất!")


🏦 BANKING INTENT CLASSIFICATION - DEMO

💬 I lost my credit card, how can I freeze it?
🏷️  Predicted: lost_or_stolen_card

💬 Why was I charged a fee for my card payment?
🏷️  Predicted: card_payment_fee_charged

💬 How do I change my PIN?
🏷️  Predicted: change_pin

💬 My transfer hasn't arrived yet.
🏷️  Predicted: transfer_not_received_by_recipient

💬 I want to close my account.
🏷️  Predicted: terminate_account

💬 Can I use Apple Pay with my card?
🏷️  Predicted: apple_pay_or_google_pay

💬 I need a refund for a transaction.
🏷️  Predicted: request_refund

💬 What is the exchange rate for USD?
🏷️  Predicted: exchange_rate

✅ Demo hoàn tất!


In [15]:
import yaml
import os

# Tạo thư mục configs và file inference.yaml y như cấu trúc project
os.makedirs("configs", exist_ok=True)
config_data = {
    "model": {
        "checkpoint_path": "outputs", # Trỏ tới thư mục chứa checkpoint đã save ở bước 8
        "base_model": "unsloth/llama-3-8b-bnb-4bit",
        "max_seq_length": 1024,
        "load_in_4bit": True
    }
}
with open("configs/inference.yaml", "w") as f:
    yaml.dump(config_data, f)
print("✅ Đã tạo file configs/inference.yaml thành công!")


✅ Đã tạo file configs/inference.yaml thành công!


In [17]:
import yaml
import torch
from unsloth import FastLanguageModel

class IntentClassification:
    def __init__(self, model_path):
        print(f"Loading configuration from {model_path}...")
        with open(model_path, "r") as f:
            self.config = yaml.safe_load(f)
            
        checkpoint_path = self.config['model']['checkpoint_path']
        print(f"Loading saved model checkpoint from '{checkpoint_path}'...")
        
        # Load lại model từ thư mục outputs (chứng minh checkpoint lưu thành công)
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=checkpoint_path,
            max_seq_length=self.config['model']['max_seq_length'],
            load_in_4bit=self.config['model']['load_in_4bit'],
        )
        FastLanguageModel.for_inference(self.model)
        
        self.prompt_template = """Below is an inquiry from a banking customer. Classify the intent of the inquiry.

### Inquiry:
{}

### Intent:
"""
        print("✅ Model loaded successfully from checkpoint!")

    def __call__(self, message):
        inputs = self.tokenizer(
            [self.prompt_template.format(message)],
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            # Generate câu trả lời
            outputs = self.model.generate(**inputs, max_new_tokens=64, use_cache=True)
        
        decoded_output = self.tokenizer.batch_decode(outputs)[0]
        
        try:
            # Tách lấy phần Intent
            predicted_label = decoded_output.split("### Intent:")[1]
            for stop_token in [self.tokenizer.eos_token, "<|end_of_text|>", "<|eot_id|>", "\n\n"]:
                if stop_token in predicted_label:
                    predicted_label = predicted_label.split(stop_token)[0]
            predicted_label = predicted_label.strip().split("\n")[0].strip()
        except Exception:
            predicted_label = "unknown"
            
        return predicted_label


In [19]:
import gc
# Dọn dẹp VRAM để chắc chắn đủ bộ nhớ load lại model
torch.cuda.empty_cache()
gc.collect()

# 1. Khởi tạo classifier với đường dẫn file config
classifier = IntentClassification(model_path="configs/inference.yaml")

# 2. Chuẩn bị một vài câu input mẫu
messages = [
    "I lost my credit card, how can I freeze it?",
    "Why was I charged a fee for my card payment?",
    "I want to open a new bank account.",
    "My debit card was stolen and I need to block it immediately."
]

print("\n" + "="*60)
print("🏦 DEMO INFERENCE - BANKING INTENT CLASSIFICATION")
print("="*60)

# 3. Chạy predict
for msg in messages:
    predicted = classifier(msg)
    print(f"\n💬 Message: {msg}")
    print(f"🏷️  Predicted Intent: {predicted}")
    
print("\n" + "="*60)


Loading configuration from configs/inference.yaml...
Loading saved model checkpoint from 'outputs'...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Model loaded successfully from checkpoint!

🏦 DEMO INFERENCE - BANKING INTENT CLASSIFICATION

💬 Message: I lost my credit card, how can I freeze it?
🏷️  Predicted Intent: lost_or_stolen_card

💬 Message: Why was I charged a fee for my card payment?
🏷️  Predicted Intent: card_payment_fee_charged

💬 Message: I want to open a new bank account.
🏷️  Predicted Intent: account_opening

💬 Message: My debit card was stolen and I need to block it immedia